# Notebook 01 - Data Collection via APIs

**Topic:** Data Collection Strategies  
**Task:** Call the Open-Meteo or REST Countries API, fetch data, and save it as a CSV  

---

## Things to learn

- How HTTP requests work in Python using the `requests` library
- How to read and parse JSON responses
- How to handle errors and status codes gracefully
- How to save API data into a structured CSV file using `pandas`

---

Both APIs used in this notebook are **completely free and require no API key**.

---

## Section 1 - Imports and Setup

In [1]:
import requests
import pandas as pd
import json
import time
import os

# Create an output folder for saved files
os.makedirs("output", exist_ok=True)

print("Libraries imported successfully.")

Libraries imported successfully.


---

## Section 2 - Understanding an HTTP Request

Every API call follows the same basic pattern:

```
requests.get(url, params=query_parameters, headers=auth_headers)
```

- **url** - the API endpoint you are calling
- **params** - key-value pairs appended to the URL as query strings (e.g. `?latitude=23.8`)
- **headers** - metadata sent with the request, commonly used for authentication

The response object contains:
- `response.status_code` - whether the request succeeded (200 = OK)
- `response.json()` - the response body parsed as a Python dictionary
- `response.text` - the raw response as a string

In [2]:
# A minimal example to see the raw structure of a response

url = "https://restcountries.com/v3.1/name/bangladesh"
response = requests.get(url)

print("Status code :", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("\nRaw JSON (first 500 chars):")
print(response.text[:500])

Status code : 200
Content-Type: application/json

Raw JSON (first 500 chars):
[{"tld":[".bd"],"cca2":"BD","ccn3":"050","cca3":"BGD","cioc":"BAN","independent":true,"status":"officially-assigned","unMember":true,"idd":{"root":"+8","suffixes":["80"]},"capital":["Dhaka"],"altSpellings":["BD","People's Republic of Bangladesh","Gônôprôjatôntri Bangladesh"],"region":"Asia","subregion":"Southern Asia","landlocked":false,"borders":["MMR","IND"],"area":147570.0,"maps":{"googleMaps":"https://goo.gl/maps/op6gmLbHcvv6rLhH6","openStreetMaps":"https://www.openstreetmap.org/relation/184


---

## Section 3 - REST Countries API

### 3.1 Fetch all countries

The REST Countries API (`restcountries.com`) provides data on every country in the world. No authentication is needed.

Endpoint: `https://restcountries.com/v3.1/all?fields=...` (filtered fields are now required)

In [ ]:
def fetch_all_countries():
    """
    Fetch data for all countries from the REST Countries API.
    Returns a list of country dictionaries on success, or None on failure.
    """
    url = "https://restcountries.com/v3.1/all"
    fields = "name,region,subregion,population,area,capital,currencies,languages,latlng,landlocked"
    headers = {"User-Agent": "axiler-week2-notebook/1.0"}

    try:
        response = requests.get(url, params={"fields": fields}, headers=headers, timeout=10)
        response.raise_for_status()  # Raises an exception for 4xx/5xx codes
        return response.json()

    except requests.exceptions.Timeout:
        print("Error: The request timed out.")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
    except requests.exceptions.ConnectionError:
        print("Error: Could not connect. Check your internet connection.")

    return None


countries_raw = fetch_all_countries()

if countries_raw:
    print(f"Successfully fetched data for {len(countries_raw)} countries.")
    print("\nKeys available for each country:")
    print(list(countries_raw[0].keys()))

HTTP error: 400 Client Error: Bad Request for url: https://restcountries.com/v3.1/all


### 3.2 Extract relevant fields

The raw JSON for each country is deeply nested. We extract only the fields we need and flatten them into a simple table.

In [ ]:
def parse_country(country):
    """
    Extract and flatten relevant fields from a single country record.
    Uses .get() with defaults to handle missing fields gracefully.
    """
    return {
        "common_name"    : country.get("name", {}).get("common", "N/A"),
        "official_name"  : country.get("name", {}).get("official", "N/A"),
        "region"         : country.get("region", "N/A"),
        "subregion"      : country.get("subregion", "N/A"),
        "population"     : country.get("population", 0),
        "area_km2"       : country.get("area", None),
        "capital"        : ", ".join(country.get("capital", [])),
        "currencies"     : ", ".join(country.get("currencies", {}).keys()),
        "languages"      : ", ".join(country.get("languages", {}).values()),
        "un_member"      : country.get("unMember", False),
        "independent"    : country.get("independent", False),
        "landlocked"     : country.get("landlocked", False),
        "latitude"       : country.get("latlng", [None, None])[0],
        "longitude"      : country.get("latlng", [None, None])[1],
    }


records = [parse_country(c) for c in countries_raw]
df_countries = pd.DataFrame(records)

print(f"DataFrame shape: {df_countries.shape}")
df_countries.head()

### 3.3 Quick inspection

In [ ]:
print("Data types:")
print(df_countries.dtypes)
print("\nMissing values:")
print(df_countries.isnull().sum())

In [ ]:
# Distribution of countries by region
print("Countries per region:")
print(df_countries["region"].value_counts())

In [ ]:
# Top 10 most populous countries
df_countries.nlargest(10, "population")[["common_name", "region", "population"]]

### 3.4 Save to CSV

In [ ]:
output_path = "output/countries.csv"
df_countries.to_csv(output_path, index=False)
print(f"Saved {len(df_countries)} rows to '{output_path}'")

---

## Section 4 - Open-Meteo Weather API

### 4.1 Fetch current weather for multiple cities

Open-Meteo (`open-meteo.com`) provides free weather forecasts and historical data. No API key is required.

We will fetch the current weather for a list of cities and collect the hourly temperature forecast for each.

In [ ]:
# Cities to query: name, latitude, longitude
CITIES = [
    {"name": "Dhaka",       "lat": 23.8103, "lon": 90.4125},
    {"name": "London",      "lat": 51.5074, "lon": -0.1278},
    {"name": "New York",    "lat": 40.7128, "lon": -74.0060},
    {"name": "Tokyo",       "lat": 35.6762, "lon": 139.6503},
    {"name": "Sydney",      "lat": -33.8688, "lon": 151.2093},
]

BASE_URL = "https://api.open-meteo.com/v1/forecast"


def fetch_weather(city):
    """
    Fetch hourly temperature and current weather for a single city.
    Returns a flat dictionary with city metadata and weather values.
    """
    params = {
        "latitude"        : city["lat"],
        "longitude"       : city["lon"],
        "current_weather" : True,
        "hourly"          : "temperature_2m,precipitation_probability,windspeed_10m",
        "forecast_days"   : 1,
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        current = data["current_weather"]

        return {
            "city"              : city["name"],
            "latitude"          : city["lat"],
            "longitude"         : city["lon"],
            "temperature_c"     : current["temperature"],
            "windspeed_kmh"     : current["windspeed"],
            "weather_code"      : current["weathercode"],
            "is_day"            : bool(current["is_day"]),
            "forecast_time"     : data["hourly"]["time"][0],
        }

    except Exception as e:
        print(f"Failed to fetch weather for {city['name']}: {e}")
        return None


weather_records = []

for city in CITIES:
    result = fetch_weather(city)
    if result:
        weather_records.append(result)
        print(f"Fetched: {city['name']} -> {result['temperature_c']} C")
    time.sleep(0.3)  # Respect the API rate limit

print(f"\nTotal cities fetched: {len(weather_records)}")

### 4.2 Build and inspect the DataFrame

In [ ]:
df_weather = pd.DataFrame(weather_records)
print(df_weather.dtypes)
df_weather

### 4.3 Fetch and save hourly forecast data for one city

This demonstrates collecting a time-series from an API, which is a very common real-world task.

In [ ]:
def fetch_hourly_forecast(city, forecast_days=7):
    """
    Fetch a multi-day hourly forecast for a given city.
    Returns a DataFrame with one row per hour.
    """
    params = {
        "latitude"      : city["lat"],
        "longitude"     : city["lon"],
        "hourly"        : "temperature_2m,precipitation_probability,windspeed_10m",
        "forecast_days" : forecast_days,
    }

    response = requests.get(BASE_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame({
        "city"                        : city["name"],
        "time"                        : pd.to_datetime(data["hourly"]["time"]),
        "temperature_c"               : data["hourly"]["temperature_2m"],
        "precipitation_probability"   : data["hourly"]["precipitation_probability"],
        "windspeed_kmh"               : data["hourly"]["windspeed_10m"],
    })

    return df


dhaka = {"name": "Dhaka", "lat": 23.8103, "lon": 90.4125}
df_hourly = fetch_hourly_forecast(dhaka, forecast_days=7)

print(f"Rows: {len(df_hourly)} (7 days x 24 hours = 168 expected)")
df_hourly.head(10)

In [ ]:
# Save both datasets
df_weather.to_csv("output/weather_current.csv", index=False)
df_hourly.to_csv("output/weather_hourly_dhaka.csv", index=False)

print("Saved weather_current.csv")
print("Saved weather_hourly_dhaka.csv")

---

## Section 5 - Handling Pagination

Many real-world APIs return results in pages. You must loop to collect all of them.

This example uses the JSONPlaceholder API which is a free fake REST API designed for practice.

In [ ]:
def fetch_paginated(base_url, page_size=10, max_pages=10):
    """
    Fetch all results from a paginated API endpoint.

    Parameters
    ----------
    base_url  : str  - The API endpoint URL
    page_size : int  - Number of results per page
    max_pages : int  - Safety cap to prevent infinite loops

    Returns
    -------
    list of dicts
    """
    all_results = []

    for page in range(1, max_pages + 1):
        params = {"_page": page, "_limit": page_size}
        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()

        batch = response.json()

        if not batch:  # Empty response means no more data
            print(f"No more results after page {page - 1}.")
            break

        all_results.extend(batch)
        print(f"Page {page}: fetched {len(batch)} records")
        time.sleep(0.2)

    return all_results


posts = fetch_paginated("https://jsonplaceholder.typicode.com/posts", page_size=10)
df_posts = pd.DataFrame(posts)

print(f"\nTotal posts collected: {len(df_posts)}")
df_posts.head()

In [ ]:
df_posts.to_csv("output/posts.csv", index=False)
print("Saved posts.csv")

---

## Section 6 - Summary and Key Takeaways

| Concept | What you did |
|---|---|
| HTTP GET request | Fetched data from REST Countries and Open-Meteo |
| JSON parsing | Extracted nested fields using `.get()` with defaults |
| Error handling | Used `try/except` and `raise_for_status()` |
| Pagination | Looped through pages until the response was empty |
| CSV export | Saved all DataFrames with `df.to_csv()` |

**Files saved in the `output/` folder:**
- `countries.csv` - 250 countries with demographic data
- `weather_current.csv` - current weather for 5 cities
- `weather_hourly_dhaka.csv` - 7-day hourly forecast for Dhaka
- `posts.csv` - paginated posts from JSONPlaceholder

---

**Practice challenge:** Try modifying `CITIES` to add 3 more cities of your choice and re-run Section 4.